# 13 - Data-Calibrated CRI (DC-CRI): Leave-One-Scenario-Out Calibration and Generalization

**Question:** can the CRI weights $\boldsymbol{\omega}$ and supervisory thresholds $\eta_A,\eta_M,\eta_D$ be
*selected from data* -- using the risk-constrained calibration procedure of
`CRI_Main_Paper_V2.tex` Sec. "Risk-Aware Calibration of CRI Parameters" -- and still generalize to a
scenario that never contributed to their own calibration?

This notebook implements Algorithm 1 (LOSO Risk Calibration and Evaluation of the CRI) exactly:
for each of the eight scenarios in turn, the CRI weights and thresholds are calibrated *only* from
the other seven scenarios' data (Eq. `loso_split`), frozen, and then evaluated on a **fresh,
genuinely closed-loop** simulation of the held-out scenario -- not a replay of an existing log,
since the activation decision itself changes the plant trajectory.

All machinery (finite-horizon label, weight QP, risk-constrained threshold search) lives in
`cri_microgrid/dc_calibration.py`; `cri.py`, `sensitivity.py`, `probing.py`, and `sim.py` are used
completely unmodified -- DC-CRI is only a different `CRIParams`, not a different architecture.

Outputs: `results/results_dc_cri_loso_folds.csv` (Package 1), `results/results_dc_cri_loso_aggregate.csv`
and `results/results_dc_cri_loso_perscenario.csv` (Package 2A/2B), `results/results_dc_cri_calibration_diagnostics.csv`
(feasibility/runtime), plus the raw calibration trajectories (`results/dc_cri_calibration_trajectories.npz`)
reused by notebook 14.

In [1]:
import sys, os, time
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd

from cri_microgrid.config import (PlantParams, ControllerParams, ConstraintParams, CRIParams,
                                   ProbeParams, ProfileParams, SimulationParams)
from cri_microgrid import scenarios, sim, metrics, validation, plotstyle
from cri_microgrid import dc_calibration as dcc

plotstyle.apply_style()
RESULTS_DIR = os.path.abspath("../results")
os.makedirs(RESULTS_DIR, exist_ok=True)

pp = PlantParams()
cp = ControllerParams()
cons = ConstraintParams()
crip = CRIParams()          # EC-CRI defaults: omega=(0.20,0.10,0.30,0.15,0.25), eta=(0.64,0.56,0.40)
probep = ProbeParams()
prof = ProfileParams()

# --- Principal calibration settings (documented choices, not prescribed by the paper) ---
T_H_PRINCIPAL = 0.5                       # s -- see notebook 14 for the T_H in {0.1,0.25,0.5,1.0} s sweep
H_PRINCIPAL = dcc.samples_to_horizon(T_H_PRINCIPAL, pp.Ts)
EPS_RISK_PRINCIPAL = 0.01                 # see notebook 14: eps_risk in {0.01,0.025,0.05,0.10} was swept
# first; 0.01 is the smallest (strictest) tested value and is the only one giving good LOSO
# generalization on most folds -- 0.025/0.05/0.10 were tried first and shown (notebook 14) to
# sit past a sharp eta_A generalization cliff for this calibration pool. See the risk-tolerance
# sensitivity table for the full comparison and the honest discussion of this finding.
LAMBDA_H = 2.0                            # hazard-penalty weight in Eq. weight_calibration (paper requires >= 1)
N_SEED_CAL = 5                            # seeds pooled per scenario for the *calibration* dataset
N_SEED_EVAL_NOISY = 5                     # extra held-out-evaluation seeds for the two noisy scenarios (S6, S8)

print(f"H = {H_PRINCIPAL} samples for T_H = {T_H_PRINCIPAL} s at Ts = {pp.Ts} s")
print(f"eps_risk = {EPS_RISK_PRINCIPAL}, lambda_H = {LAMBDA_H}, N_SEED_CAL = {N_SEED_CAL}")

H = 25 samples for T_H = 0.5 s at Ts = 0.02 s
eps_risk = 0.01, lambda_H = 2.0, N_SEED_CAL = 5


## Scenario builders and the finite-horizon ground truth

The finite-horizon hazard label $z_k^{(H)}$ (Eq. `finite_horizon_hazard`) uses the hard operational
safe set $\mathcal{Q}_{\mathrm{s}}$: true bus voltages, feeder currents, and DC-link voltage only
(`dc_calibration.hard_limit_violation`) -- not the broader multi-mechanism label used by the
original EC-CRI `N_FA` numbers elsewhere in the paper (weak authority, noise, drift are diagnostic
mechanism labels, Eq. `mechanism_hazard`, not the activation-risk label used for calibration).
This is a documented modeling choice -- see the `LaTeX_Edit_Add_V2.ipynb` notes.

In [2]:
SCENARIO_BUILDERS = {
    "S1": lambda: scenarios.build_S1(pp, prof, T_end=40.0),
    "S2": lambda: scenarios.build_S2(pp, prof, T_end=40.0),
    "S3": lambda: scenarios.build_S3(pp, prof, T_end=40.0),
    "S4": lambda: scenarios.build_S4(pp, prof, T_end=40.0),
    "S5": lambda: scenarios.build_S5(pp, prof, ramp_kind="fast", T_end=40.0),
    "S6": lambda: scenarios.build_S6(pp, prof, noise_level_pct=1.0, T_end=40.0),
    "S7": lambda: scenarios.build_S7(pp, prof, T_end=40.0, kind="vsc2_outage"),
    "S8": lambda: scenarios.build_S8(pp, prof, T_end=50.0),
}
NOISY_SCENARIOS = {"S6", "S8"}
SCENARIO_ORDER = list(SCENARIO_BUILDERS.keys())

## Step 1: generate the calibration dataset (EC-CRI-supervised runs, multiple seeds per scenario)

$\Gamma_k=[\RI_k,\CI_k,\SMI_k,\AIi_k,\NCI_k]$ does **not** depend on $\boldsymbol{\omega}$ or
$\eta$ -- only the downstream combination and thresholds do -- so these readiness-vector
trajectories can be generated once, from an already-validated EC-CRI closed loop (exactly as a
real commissioning dataset would be collected from historical operation), and reused for every
LOSO fold's weight/threshold calibration below.

In [3]:
t0 = time.time()
cal_trajectories = {name: [] for name in SCENARIO_ORDER}
for name, builder in SCENARIO_BUILDERS.items():
    scenario = builder()
    for seed in range(N_SEED_CAL):
        simp = SimulationParams(T_end=scenario.T_end, seed=seed, strategy="cri")
        lg = sim.run_closed_loop(scenario, pp, cp, cons, crip, probep, simp)
        cal_trajectories[name].append(dcc.collect_trajectory(lg, cons, name, seed, pp.Ts))

print(f"Generated {N_SEED_CAL} calibration seeds x {len(SCENARIO_ORDER)} scenarios in {time.time()-t0:.1f} s")

# Save the raw trajectories so notebook 14 does not need to re-simulate.
np.savez(
    os.path.join(RESULTS_DIR, "dc_cri_calibration_trajectories.npz"),
    **{f"{name}_seed{t.seed}_Gamma": t.Gamma for name in SCENARIO_ORDER for t in cal_trajectories[name]},
    **{f"{name}_seed{t.seed}_sigma": t.sigma_hat_sq for name in SCENARIO_ORDER for t in cal_trajectories[name]},
    **{f"{name}_seed{t.seed}_viol": t.viol for name in SCENARIO_ORDER for t in cal_trajectories[name]},
)
print("Saved raw calibration trajectories to results/dc_cri_calibration_trajectories.npz")

Generated 5 calibration seeds x 8 scenarios in 32.0 s
Saved raw calibration trajectories to results/dc_cri_calibration_trajectories.npz


## Step 2: the 8-fold LOSO loop (Algorithm 1)

For each held-out scenario: pool the other seven scenarios' calibration trajectories, solve the
weight QP (Eq. `weight_calibration`) and the risk-constrained threshold search (Eq.
`risk_constrained_threshold`), freeze the result into a `CRIParams`, then run a **fresh** closed-loop
evaluation of all four strategies (Direct, Hard-limit, EC-CRI, DC-CRI) on the held-out scenario.

In [4]:
def evaluate_strategy_on_scenario(scenario_name, builder, strategy, crip_eval, seeds):
    """Run `strategy` on `scenario_name` for each seed in `seeds`, returning a
    list of per-seed metric dicts computed under the finite-horizon label."""
    rows = []
    for seed in seeds:
        scenario = builder()
        simp = SimulationParams(T_end=scenario.T_end, seed=seed, strategy=strategy)
        lg = sim.run_closed_loop(scenario, pp, cp, cons, crip_eval, probep, simp)
        z_H = dcc.finite_horizon_label(lg, cons, H_PRINCIPAL)
        rows.append(dict(
            seed=seed,
            N_FA=metrics.false_activation_count_horizon(lg["s"], z_H),
            FAR=metrics.far(lg["s"], z_H),
            FDR=metrics.fdr(lg["s"], z_H),
            A_c=metrics.controller_availability(lg["s"]),
            E_V=metrics.voltage_deviation(lg["V1"], lg["V2"], pp.Vac_nom),
        ))
    return rows


fold_rows = []           # Package 1: weights/thresholds per fold
perscenario_rows = []    # Package 2B: per-scenario, per-strategy performance
diagnostics_rows = []    # Calibration feasibility/runtime

t_loso0 = time.time()
for held_out in SCENARIO_ORDER:
    cal_pool = [t for name in SCENARIO_ORDER if name != held_out for t in cal_trajectories[name]]

    outcome = dcc.calibrate_dc_cri(cal_pool, H=H_PRINCIPAL, eps_risk=EPS_RISK_PRINCIPAL,
                                    lambda_H=LAMBDA_H, crip_template=crip)
    crip_dc = dcc.dc_cri_params_from_outcome(outcome, crip)

    fold_rows.append(dict(
        held_out=held_out,
        omega_R=outcome.omega[0], omega_C=outcome.omega[1], omega_S=outcome.omega[2],
        omega_A=outcome.omega[3], omega_N=outcome.omega[4],
        eta_A=outcome.eta_A, eta_M=outcome.eta_M, eta_D=outcome.eta_D,
        delta_M=outcome.delta_M, delta_D=outcome.delta_D,
    ))

    # Held-out evaluation seeds: representative seed 1 always; extra seeds only
    # matter for the two scenarios with measurement noise (S1-S5,S7 are exactly
    # deterministic across seeds, verified in notebooks 08/10).
    eval_seeds = [1] + (list(range(N_SEED_EVAL_NOISY)) if held_out in NOISY_SCENARIOS else [])
    eval_seeds = sorted(set(eval_seeds))
    builder = SCENARIO_BUILDERS[held_out]

    strategy_params = {
        "Direct": ("direct", crip),
        "Hard-limit": ("threshold", crip),
        "EC-CRI": ("cri", crip),
        "DC-CRI": ("cri", crip_dc),
    }
    test_far_for_diag = None
    for method, (strat, crip_use) in strategy_params.items():
        rows = evaluate_strategy_on_scenario(held_out, builder, strat, crip_use, eval_seeds)
        for r in rows:
            perscenario_rows.append(dict(held_out=held_out, method=method, **r))
        if method == "DC-CRI":
            test_far_for_diag = np.mean([r["FAR"] for r in rows])

    diagnostics_rows.append(dict(
        held_out=held_out, feasible=outcome.feasible,
        weight_runtime_s=outcome.weight_result.runtime_s,
        weight_n_iter=outcome.weight_result.n_iter, weight_n_feval=outcome.weight_result.n_feval,
        weight_objective=outcome.weight_result.objective, weight_success=outcome.weight_result.success,
        n_safe_samples=outcome.weight_result.n_safe, n_hazard_samples=outcome.weight_result.n_hazard,
        risk_constraint_active=bool(outcome.feasible and outcome.cal_risk >= 0.9 * EPS_RISK_PRINCIPAL),
        cal_FAR=outcome.cal_risk, cal_availability=outcome.cal_availability,
        test_FAR=test_far_for_diag,
    ))
    print(f"[{held_out}] feasible={outcome.feasible}  eta_A*={outcome.eta_A:.4f}  "
          f"omega*={np.round(outcome.omega,3)}  cal_FAR={outcome.cal_risk:.4f}  test_FAR(DC-CRI)={test_far_for_diag:.4f}")

print(f"\nFull 8-fold LOSO (calibration + 4-strategy evaluation) completed in {time.time()-t_loso0:.1f} s")

[S1] feasible=True  eta_A*=0.5740  omega*=[0.075 0.    0.759 0.092 0.074]  cal_FAR=0.0098  test_FAR(DC-CRI)=0.0000
[S2] feasible=True  eta_A*=0.5740  omega*=[0.045 0.    0.874 0.081 0.   ]  cal_FAR=0.0100  test_FAR(DC-CRI)=0.0272
[S3] feasible=True  eta_A*=0.5680  omega*=[0.134 0.    0.775 0.02  0.072]  cal_FAR=0.0100  test_FAR(DC-CRI)=0.0000
[S4] feasible=True  eta_A*=0.5720  omega*=[0.075 0.    0.757 0.09  0.077]  cal_FAR=0.0099  test_FAR(DC-CRI)=0.0000
[S5] feasible=True  eta_A*=0.3500  omega*=[0.073 0.    0.736 0.09  0.1  ]  cal_FAR=0.0097  test_FAR(DC-CRI)=0.6784
[S6] feasible=True  eta_A*=0.4600  omega*=[0.087 0.    0.487 0.08  0.346]  cal_FAR=0.0099  test_FAR(DC-CRI)=0.0000
[S7] feasible=True  eta_A*=0.3480  omega*=[0.    0.    0.693 0.155 0.152]  cal_FAR=0.0099  test_FAR(DC-CRI)=0.0320
[S8] feasible=True  eta_A*=0.5780  omega*=[0.473 0.    0.273 0.033 0.22 ]  cal_FAR=0.0099  test_FAR(DC-CRI)=1.0000

Full 8-fold LOSO (calibration + 4-strategy evaluation) completed in 37.7 s


## Package 1: eight LOSO calibrated weight vectors and thresholds

In [5]:
fold_df = pd.DataFrame(fold_rows).set_index("held_out")
fold_df.to_csv(os.path.join(RESULTS_DIR, "results_dc_cri_loso_folds.csv"))
display(fold_df.round(4))

print("\nFold-to-fold stability (mean +/- std across the 8 folds):")
display(fold_df.agg(["mean", "std", "min", "max"]).round(4))

,omega_R,omega_C,omega_S,omega_A,omega_N,eta_A,eta_M,eta_D,delta_M,delta_D
held_out,,,,,,,,,,
S1,0.0749,0.0,0.7591,0.0916,0.0744,0.574,0.454,0.254,0.12,0.32
S2,0.0446,0.0,0.8741,0.0813,0.0000,0.574,0.454,0.294,0.12,0.28
S3,0.1336,0.0,0.7747,0.0200,0.0717,0.568,0.448,0.248,0.12,0.32
S4,0.0754,0.0,0.7575,0.0902,0.0769,0.572,0.452,0.252,0.12,0.32
S5,0.0734,0.0,0.7357,0.0904,0.1004,0.350,0.230,0.070,0.12,0.28
S6,0.0870,0.0,0.4870,0.0797,0.3463,0.460,0.380,0.140,0.08,0.32
S7,0.0000,0.0,0.6929,0.1552,0.1519,0.348,0.268,0.068,0.08,0.28
S8,0.4735,0.0,0.2733,0.0331,0.2201,0.578,0.478,0.458,0.10,0.12



Fold-to-fold stability (mean +/- std across the 8 folds):


,omega_R,omega_C,omega_S,omega_A,omega_N,eta_A,eta_M,eta_D,delta_M,delta_D
mean,0.1203,0.0,0.6693,0.0802,0.1302,0.5030,0.3955,0.2230,0.1075,0.2800
std,0.1476,0.0,0.1942,0.0411,0.1085,0.1028,0.0952,0.1293,0.0183,0.0676
min,0.0000,0.0,0.2733,0.0200,0.0000,0.3480,0.2300,0.0680,0.0800,0.1200
max,0.4735,0.0,0.8741,0.1552,0.3463,0.5780,0.4780,0.4580,0.1200,0.3200


## Package 2: Direct vs. Hard-limit vs. EC-CRI vs. DC-CRI under LOSO

In [6]:
perscenario_df = pd.DataFrame(perscenario_rows)
perscenario_df.to_csv(os.path.join(RESULTS_DIR, "results_dc_cri_loso_perscenario.csv"), index=False)

# 2B: per (held-out scenario, method), mean over evaluation seeds
per_scenario_summary = perscenario_df.groupby(["held_out", "method"])[["N_FA", "FAR", "FDR", "A_c", "E_V"]].mean()
per_scenario_summary = per_scenario_summary.reindex(SCENARIO_ORDER, level="held_out")
per_scenario_summary.to_csv(os.path.join(RESULTS_DIR, "results_dc_cri_loso_perscenario_summary.csv"))
print("Package 2B -- per held-out scenario (mean over evaluation seeds):")
display(per_scenario_summary.round(4))

# 2A: aggregate across all 8 held-out folds, each scenario weighted equally.
# IMPORTANT: this must NOT be a plain mean over perscenario_df rows, since S6
# and S8 (NOISY_SCENARIOS) contribute N_SEED_EVAL_NOISY=5 rows each while every
# other scenario contributes exactly 1 -- pooling raw rows would silently give
# S6/S8 5x the weight of every other scenario. Instead: average seeds within each
# scenario first (per_scenario_summary, above), THEN average equally across the
# 8 resulting scenario-level values. This matches the convention used by notebook
# 14 risk/horizon sensitivity sweeps (run_full_loso), so Table tab:loso_results and
# Table tab:risk_sensitivity/tab:horizon_sensitivity are now directly comparable
# at eps_risk=0.01, T_H=0.5 s.
method_order = ["Direct", "Hard-limit", "EC-CRI", "DC-CRI"]
scenario_level = per_scenario_summary.reset_index()
agg_mean = scenario_level.groupby("method")[["N_FA", "FAR", "FDR", "A_c", "E_V"]].mean().reindex(method_order)
agg_std = scenario_level.groupby("method")[["N_FA", "FAR", "FDR", "A_c", "E_V"]].std().reindex(method_order)
agg_df = agg_mean.join(agg_std, rsuffix="_std")
agg_df.to_csv(os.path.join(RESULTS_DIR, "results_dc_cri_loso_aggregate.csv"))
print()
print("Package 2A -- aggregate across all 8 held-out folds (mean +/- std of the 8 equally weighted scenario-level values):")
display(agg_df.round(4))


Package 2B -- per held-out scenario (mean over evaluation seeds):


N_FA     FAR     FDR     A_c     E_V
held_out method                                            
S1       DC-CRI         0.0  0.0000  0.0005  0.9995  0.0023
         Direct         0.0  0.0000  0.0000  1.0000  0.0018
         EC-CRI         0.0  0.0000  0.0015  0.9985  0.0023
         Hard-limit     0.0  0.0000  0.0000  1.0000  0.0018
S2       DC-CRI        23.0  0.0272  0.0000  0.5887  0.0512
         Direct         0.0  0.0000  0.0000  1.0000  0.0161
         EC-CRI         0.0  0.0000  0.0015  0.9985  0.0197
         Hard-limit     0.0  0.0000  0.0000  1.0000  0.0161
S3       DC-CRI         0.0  0.0000  0.3783  0.6217  0.0098
         Direct      1516.0  1.0000  0.0000  1.0000  0.0129
         EC-CRI         0.0  0.0000  0.4038  0.5962  0.0105
         Hard-limit   291.0  0.2233  0.0029  0.4933  0.0153
S4       DC-CRI         0.0  0.0000  0.0005  0.9995  0.0067
         Direct         0.0  0.0000  0.0000  1.0000  0.0064
         EC-CRI         0.0  0.0000  0.0015  0.9985  0.0069
         Hard-limit     0.0  0.0000  0.0000  1.0000  0.0064
S5       DC-CRI       831.0  0.6784  0.0361  0.7891  0.0098
         Direct      1487.0  1.0000  0.0000  1.0000  0.0107
         EC-CRI        20.0  0.2410  0.5704  0.4218  0.0165
         Hard-limit  1091.0  0.7048  0.0243  0.7661  0.0090
S6       DC-CRI         0.0  0.0000  0.0000  1.0000  0.0108
         Direct         0.0  0.0000  0.0000  1.0000  0.0107
         EC-CRI         0.0  0.0000  0.0015  0.9985  0.0108
         Hard-limit     0.0  0.0000  0.0001  0.9999  0.0107
S7       DC-CRI        36.0  0.0320  0.0000  0.4558  0.1288
         Direct      1125.0  1.0000  0.0000  1.0000  0.1302
         EC-CRI        29.0  0.0258  0.0034  0.4508  0.1288
         Hard-limit    26.0  0.0231  0.0000  0.4508  0.1286
S8       DC-CRI      2104.4  1.0000  0.0150  0.9977  0.1624
         Direct      2231.4  1.0000  0.0000  1.0000  0.1560
         EC-CRI         0.0  0.0000  0.9956  0.0015  0.1688
         Hard-limit   368.6  0.1887  0.1466  0.3351  0.1660


Package 2A -- aggregate across all 8 held-out folds (mean +/- std of the 8 equally weighted scenario-level values):


,N_FA,FAR,FDR,A_c,E_V,N_FA_std,FAR_std,FDR_std,A_c_std,E_V_std
method,,,,,,,,,,
Direct,794.925,0.5000,0.0000,1.0000,0.0431,902.2947,0.5345,0.0000,0.0000,0.0623
Hard-limit,222.075,0.1425,0.0217,0.7556,0.0442,381.1572,0.2450,0.0511,0.2872,0.0645
EC-CRI,6.125,0.0333,0.2474,0.6830,0.0455,11.5936,0.0844,0.3761,0.3765,0.0649
DC-CRI,374.300,0.2172,0.0538,0.8065,0.0477,755.9366,0.3936,0.1317,0.2247,0.0629


## Calibration feasibility and runtime diagnostics

In [7]:
diag_df = pd.DataFrame(diagnostics_rows).set_index("held_out")
diag_df.to_csv(os.path.join(RESULTS_DIR, "results_dc_cri_calibration_diagnostics.csv"))
display(diag_df.round(5))

n_feasible = int(diag_df["feasible"].sum())
print(f"\nCalibration feasibility: {n_feasible}/8 folds feasible at eps_risk={EPS_RISK_PRINCIPAL}.")
print(f"Mean weight-calibration runtime: {diag_df['weight_runtime_s'].mean()*1000:.2f} ms/fold "
      f"({diag_df['weight_n_iter'].mean():.1f} mean SLSQP iterations).")

,feasible,weight_runtime_s,weight_n_iter,weight_n_feval,weight_objective,weight_success,n_safe_samples,n_hazard_samples,risk_constraint_active,cal_FAR,cal_availability,test_FAR
held_out,,,,,,,,,,,,
S1,True,0.00290,10,10,0.29472,True,58357,14178,True,0.00980,0.52073,0.00000
S2,True,0.00268,9,9,0.25259,True,58357,14178,True,0.00997,0.58056,0.02719
S3,True,0.00314,11,11,0.24333,True,58357,14178,True,0.00998,0.58715,0.00000
S4,True,0.00277,10,10,0.29398,True,58357,14178,True,0.00988,0.52307,0.00000
S5,True,0.00296,10,10,0.26510,True,58772,13763,True,0.00969,0.69540,0.67837
S6,True,0.00283,11,11,0.28705,True,58357,14178,True,0.00995,0.67212,0.00000
S7,True,0.00313,11,11,0.29124,True,63982,8553,True,0.00994,0.76693,0.03200
S8,True,0.00267,9,9,0.22702,True,63995,6040,True,0.00994,0.86885,1.00000



Calibration feasibility: 8/8 folds feasible at eps_risk=0.01.
Mean weight-calibration runtime: 2.89 ms/fold (10.1 mean SLSQP iterations).


## Takeaway

The eight LOSO folds show whether DC-CRI's calibration is stable (Package 1: do the calibrated
weights and thresholds vary wildly depending on which scenario is excluded, or do they settle into
a consistent physical structure?) and whether the resulting supervisor actually generalizes to a
scenario it never calibrated on (Package 2: does DC-CRI's out-of-scenario FAR/FDR/A_c stay close to
EC-CRI's, or does removing engineering judgment from threshold selection cost something in
unseen-scenario safety or availability?). See `LaTeX_Edit_Add_V2.ipynb` for the numeric answers to
both questions and the exact paper text/tables built from these results.